<img align="right" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="left">Model Inference</h1>
<h4 align="left">Written by the KSO Team</h4>

This notebook runs a trained YOLO model on a folder of videos and produces a single `all_detections.csv` for downstream analysis.

### Quick Start

1. Train a model using `Train+Eval_models.ipynb` (or bring your own `.pt` weights)
2. Place your video files in a single directory
3. Edit **Phase 1** with your paths and settings
4. Run **Phase 2** — cleans, infers, and writes outputs

Uses YOLO stream mode for constant memory regardless of video length. Detections are appended to the CSV after each video, so partial results are preserved if the run is interrupted.

---

### Key Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `conf_thres` | 0.5 | Minimum detection confidence |
| `vid_stride` | 1 | Process every Nth frame (higher = faster, fewer detections) |
| `img_size` | 640 | YOLO input resolution (match training if custom) |
| `CLEAN_GOPRO` | True | Strip video metadata streams that can cause errors (False to disable)|
| `PREVIEW_SECONDS` | 0 | Seconds of annotated preview video to generate (0 = disable) |

---
## Phase 1: Configuration

Edit the fields below to match your setup, then run the cell.

**`vid_stride`** controls how many frames are skipped between detections.
At stride 1 every frame is processed; at stride 60 on a 60fps video, you get roughly one detection per second.
Higher stride = faster inference, but lower temporal resolution in your output CSV.
Timestamps in the CSV always reflect real video time regardless of stride.

In [ ]:
from pathlib import Path

# ── Paths ──
model_path = Path("<path-to-best.pt>").expanduser().resolve()  # Path to model weights
videos_dir = Path("<video-folder>").expanduser().resolve()  # Your video folder
output_dir = Path("<output-folder>").expanduser().resolve()  # csv/preview outputs

# ── Inference ──
conf_thres = 0.5  # Minimum confidence threshold
vid_stride = 1  # Process every Nth frame (1 = all, 60 ≈ 1/sec at 60fps)
img_size = 640  # YOLO input size (match training if custom)

# ── Options ──
n_videos = "all"  # "all" (with quotes) or an integer like 5 (without quotes)
CLEAN_GOPRO = True  # Strip metadata streams that cause OpenCV errors
PREVIEW_SECONDS = 0  # 0 = disabled; try 10 for a 10s annotated preview

# ── Validate ──
assert (
    model_path.is_file() and model_path.suffix == ".pt"
), f"Model weights (.pt) not found: {model_path}"
assert n_videos == "all" or isinstance(
    n_videos, int
), f'n_videos must be "all" or an int, got {n_videos!r}'

print(f"Model:    {model_path.parent.parent.name}")
print(f"Videos:   {videos_dir}")
print(f"Output:   {output_dir}")
print(f"Process:  {n_videos} video(s)")
print(f"Preview:  {PREVIEW_SECONDS}s")

---
## Phase 2: Run Inference

In [ ]:
import gc, time, subprocess, shutil, sys
import torch
import cv2
import pandas as pd
from ultralytics import YOLO

# ── Validate paths ──
assert (
    model_path.is_file() and model_path.suffix == ".pt"
), f"Model weights (.pt) not found: {model_path}"
assert videos_dir.is_dir(), f"Video directory not found: {videos_dir}"

# ── Output setup ──
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "all_detections.csv"
if csv_path.exists():
    csv_path.unlink()  # Fresh file; detections are appended per video

# ── Discover videos ──
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
all_videos = sorted(
    p
    for p in videos_dir.iterdir()
    if p.is_file() and p.suffix.lower() in VIDEO_EXTS and "_clean" not in p.stem
)
if isinstance(n_videos, int):
    selected = all_videos[:n_videos]
else:
    selected = all_videos
assert len(selected) > 0, f"No video files found in {videos_dir}"
print(f"{len(selected)} video(s) in {videos_dir}")

# ── Find ffmpeg ──
ffmpeg_exe = None
if CLEAN_GOPRO:
    ffmpeg_exe = shutil.which("ffmpeg")
    if ffmpeg_exe is None:
        try:
            import imageio_ffmpeg

            ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()
        except ImportError:
            pass
    if ffmpeg_exe is None:
        print("Warning: ffmpeg not found, skipping video cleaning")

# ── Load model ──
model = YOLO(str(model_path))
names = model.names
half = torch.cuda.is_available()
print(f"Model: {model_path.name} | Classes: {names}")
print("=" * 60)

total_rows = 0
preview_done = False
csv_header_written = False

for idx, video_path in enumerate(selected, 1):

    # ── Clean if needed ── (write to output_dir so videos_dir can be read-only)
    inference_path = video_path
    if CLEAN_GOPRO and ffmpeg_exe:
        clean_dir = output_dir / "cleaned_videos"
        clean_dir.mkdir(exist_ok=True)
        cleaned = clean_dir / f"{video_path.stem}_clean.mp4"
        if cleaned.exists():
            inference_path = cleaned
        else:
            cmd = [
                ffmpeg_exe,
                "-i",
                str(video_path),
                "-map",
                "0:v:0",
                "-c",
                "copy",
                "-an",
                "-sn",
                "-map_metadata",
                "-1",
                "-y",
                str(cleaned),
            ]
            r = subprocess.run(cmd, capture_output=True)
            inference_path = cleaned if r.returncode == 0 else video_path

    # ── Video metadata ──
    cap = cv2.VideoCapture(str(inference_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )
    cap.release()
    expected = max(1, total_frames // max(vid_stride, 1))

    print(
        f"\n[{idx}/{len(selected)}] {video_path.name}  ({w}x{h}, {fps:.1f}fps, {total_frames/fps:.1f}s)"
    )

    # ── Preview setup (first video only) ──
    preview_writer = None
    preview_limit = 0
    preview_path = None
    preview_fps = None
    if PREVIEW_SECONDS > 0 and not preview_done:
        preview_dir = output_dir / "preview"
        preview_dir.mkdir(exist_ok=True)
        preview_path = preview_dir / f"{video_path.stem}_preview.mp4"
        preview_fps = max(10.0, fps / max(1, vid_stride))
        preview_limit = int(PREVIEW_SECONDS * preview_fps)

    # ── Inference loop ──
    detections = []
    frame_count = 0
    preview_frames = 0
    start = time.time()

    for result in model.predict(
        source=str(inference_path),
        conf=conf_thres,
        stream=True,
        vid_stride=vid_stride,
        imgsz=img_size,
        half=half,
        verbose=False,
    ):
        src_frame = frame_count * vid_stride
        ts = src_frame / fps

        boxes = result.boxes
        if boxes is not None and len(boxes):
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            clss = boxes.cls.int().cpu().numpy()

            for b, cf, c in zip(xyxy, confs, clss):
                detections.append(
                    {
                        "video": video_path.stem,
                        "frame": src_frame,
                        "timestamp_s": round(ts, 3),
                        "class_name": names[c],
                        "confidence": round(float(cf), 4),
                        "x1": round(float(b[0]), 1),
                        "y1": round(float(b[1]), 1),
                        "x2": round(float(b[2]), 1),
                        "y2": round(float(b[3]), 1),
                    }
                )

        # Preview (same pass, first video only)
        if preview_limit and preview_frames < preview_limit:
            ann = result.plot(line_width=2, labels=True, conf=True)
            if preview_writer is None:
                fh, fw = ann.shape[:2]
                preview_writer = cv2.VideoWriter(
                    str(preview_path),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    preview_fps,
                    (fw, fh),
                )
            preview_writer.write(ann)
            preview_frames += 1

        frame_count += 1

        # Progress
        if frame_count % 200 == 0:
            elapsed = time.time() - start
            pct = min(frame_count / expected * 100, 100)
            rate = frame_count / elapsed
            eta = (expected - frame_count) / rate if rate > 0 else 0
            sys.stdout.write(
                f"\r  {pct:5.1f}% | {frame_count}/{expected} frames | {rate:.0f} fps | ETA {eta:.0f}s  "
            )
            sys.stdout.flush()

    if preview_writer:
        preview_writer.release()
        preview_done = True

    # ── Append this video's detections to CSV (incremental write) ──
    if detections:
        pd.DataFrame(detections).to_csv(
            csv_path, mode="a", header=not csv_header_written, index=False
        )
        csv_header_written = True
    total_rows += len(detections)

    elapsed = time.time() - start
    rate = frame_count / max(elapsed, 0.1)
    sys.stdout.write(
        f"\r  {frame_count} frames | {elapsed:.1f}s | {rate:.0f} fps | {len(detections)} detections"
    )
    if preview_limit and preview_done and idx == 1:
        sys.stdout.write(f" | preview: {preview_frames}fr")
    print()

    del detections
    gc.collect()

# ── Summary ──
print("\n" + "=" * 60)
print(f"{len(selected)} video(s) — {total_rows} detections")
if csv_path.exists() and total_rows > 0:
    summary = pd.read_csv(csv_path, usecols=["class_name"])
    print(summary["class_name"].value_counts().to_string())
print(f"\n→ {csv_path}")

---
## ✅ Done

```
<output_dir>/
├── all_detections.csv          ← main output
├── cleaned_videos/             (if CLEAN_GOPRO and ffmpeg available)
│   └── <video>_clean.mp4
└── preview/
    └── <video>_preview.mp4     (if PREVIEW_SECONDS > 0)
```

Each detection row: `video, frame, timestamp_s, class_name, confidence, x1, y1, x2, y2`

### Next Steps

1. **Analyse detections** — use `all_detections.csv` for ecological metrics (maxN, detection percentages)
2. **Compare models** — change `model_path`, use a different `output_dir`, and re-run